# 9. Describing a dataset: summaries and Claude: solutions

Exercises of [chapter 9](../../09_describing_a_dataset.md). Run the setup cell first.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, difflib, json, shutil, tempfile
import rasterio
import yaml
from data_acquisition.catalog import build, describe, evidence
from data_acquisition.catalog import summary as summaries

OUT = ROOT / "output/manual/describe/training_pond"
REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())


def check(s):
    """Check a summary (a dict) and build it in memory; print what the pipeline says."""
    path = Path(tempfile.mkdtemp()) / "my_summary.yaml"
    path.write_text(yaml.safe_dump(copy.deepcopy(s), sort_keys=False))
    try:
        loaded = summaries.load(str(path))
    except ValueError as e:
        print("BLOCKS:\n", str(e).split("\n", 1)[1])
        return None
    catalog, report = build.build(SAMPLE, loaded)
    print("valid; build report:", {k: v for k, v in report.items() if v} or "empty")
    return catalog

**1. Evidence.** Gather the evidence and the prompt for the sample dataset into `output/manual/describe/training_pond`. For each file, what did code read? What did it read about the photos, and why so little?

> **From the chapter.** §2: code gathers what it can read for sure, file by file.
>
> **Code.** `describe prompt`, a command of [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py), run with `subprocess`; it gathers with [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py) and writes `evidence.json` and `prompt.md` into `OUT`.

In [ ]:
subprocess.run([sys.executable, "-m", "data_acquisition.catalog.describe", "prompt",
                "--root", str(SAMPLE), "--folders", "repo/*", "--dataset", "training_pond",
                "--source-kind", "dataone", "--out", str(OUT)], cwd=ROOT, check=True, capture_output=True)
ev = json.loads((OUT / "evidence.json").read_text())
for f in ev["folders"][0]["files"]:
    print(f"{f['name']:45s} {[k for k in f if k not in ('name', 'bytes')]}")
print("photo folders become one entry from", evidence.PHOTO_FOLDER, "photos up; the sample has 6, read one by one, "
      "and code reads nothing from a single photo")

**2. The rules.** Read the rules in `prompt.md`. Which of them stop Claude from inventing values?

> **From the chapter.** §3: the drafting rules that matter most.
>
> **Code.** `prompt.md` in `OUT`, or `describe.RULES` in [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py).

In [ ]:
for line in describe.RULES.split("\n- "):
    if any(w in line.lower() for w in ("never", "only", "invent", "unclear", "disagree")):
        print("-", " ".join(line.split())[:220])

**3. By hand.** Write a summary for the sample from the evidence alone, starting from the skeleton in the notebook; make it pass `check`; compare it with the reference summary.

> **From the chapter.** §1: the parts of a summary. [Chapter 6](../../06_what_the_pipeline_needs.md) lists every field, and what happens without it.
>
> **Code.** `check(s)`, defined above, calls `summaries.load` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py) and `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py). Compare with [`docs/manual/solutions/training_pond.yaml`](../training_pond.yaml).

In [ ]:
mine = {
    "schema": 1, "dataset": "training_pond", "title": "Training Pond drone survey, July 2023",
    "authored": {"by": "A Student", "date": datetime.date.today()}, "reviewed": None,
    "source": {"kind": "dataone", "provider": "Example repository (invented)"},
    "groups": [{"folder": "repo/doi_10.5555_training.pond", "record": {"path": "metadata.xml", "format": "eml"},
                "site": "training_pond", "positioning": ["gnss"], "files": [
        {"path": "ortho_2023-07-12.tif", "product": "orthomosaic", "date": datetime.date(2023, 7, 12)},
        {"path": "dsm_2023-07-12.tif", "product": "dsm", "date": datetime.date(2023, 7, 12)},
        {"path": "dem_2022.asc", "product": "dtm", "date": datetime.date(2022, 8, 1), "date_precision": "month",
         "crs": "EPSG:32612"},
        {"path": "dams_2023-07-12.gpkg", "product": "annotations", "date": datetime.date(2023, 7, 12)},
        {"path": "photos_2023-07-12", "product": "photos", "date": datetime.date(2023, 7, 12)},
        {"path": "README.txt", "role": "metadata", "attach_to": ["orthomosaic", "dsm", "dtm", "annotations", "photos"]},
    ]}],
}
check(mine)
ref_files = {f["path"]: f for f in REFERENCE["groups"][0]["files"]}
for f in mine["groups"][0]["files"]:
    extra = {k: v for k, v in ref_files[f["path"]].items() if k not in f}
    if extra:
        print(f"the reference also gives {f['path']}: {extra}")

**4. Review a draft.** [`docs/manual/data/flawed_draft.yaml`](../../data/flawed_draft.yaml) passes the check but has four mistakes a reviewer should catch. Find them with the evidence.

> **From the chapter.** §4: a draft that passes the check is valid, not yet right. The review compares it with the evidence.
>
> **Code.** [`docs/manual/data/flawed_draft.yaml`](../../data/flawed_draft.yaml), `check(s)` defined above, and the evidence of exercise 1.

In [ ]:
flawed = yaml.safe_load((ROOT / "docs/manual/data/flawed_draft.yaml").read_text())
check(flawed)
files = {f["path"]: f for f in flawed["groups"][0]["files"]}
with rasterio.open(DATASET / "ortho_2023-07-12.tif") as r:
    print("1. bands: draft", files["ortho_2023-07-12.tif"]["bands"], "| file declares", [c.name for c in r.colorinterp])
methods = (DATASET / "metadata.xml").read_text()
print("2. dem_2022.asc: draft date", files["dem_2022.asc"]["date"], "| methods text:",
      methods[methods.find("The 2022"):methods.find("The 2022") + 90])
print("3. positioning: draft", flawed["groups"][0]["positioning"], "| nothing in the evidence mentions RTK")
print("4. dams: draft product", files["dams_2023-07-12.gpkg"]["product"], "| methods: 'Dams were mapped by hand on the "
      "orthomosaic' -> annotations, not surveyed points")
catalog = check(flawed)
ortho = next(i for i in catalog.get_items(recursive=True) if i.id.endswith("orthomosaic"))
print("the build flags the bands:", [q["flag"] for q in ortho.properties["beaver:quality"]])

**5. Load and check.** Load the reference summary with `summaries.load`: what is added to it? Then run `summaries.check` on the file as written and on the loaded summary, and on copies with a missing title, an unknown product and an unknown field. What does `check` expect?

> **From the chapter.** §5, "`load` changes the summary": `check` must be given the summary as written.
>
> **Code.** `summaries.load` and `summaries.check`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
path = ROOT / "docs/manual/solutions/training_pond.yaml"
s = summaries.load(str(path))                       # read, validated, and completed
print("keys:", sorted(s))
print("sites:", s["sites"]["training_pond"]["name"], "| path:", Path(s["_path"]).name)
raw = yaml.safe_load(path.read_text())              # the file as written, nothing added
print("check, as written:", summaries.check(raw))
print("check, as loaded: ", summaries.check(s))
# check expects the summary as written. load replaces `sites` with the registry's full entries, and a summary may
# only give hints there, so check refuses the loaded one. load calls check itself, so it never needs repeating.

no_title = copy.deepcopy(raw)
del no_title["title"]
print("no title:", summaries.check(no_title))      # only this: a missing top-level field stops the check early
wrong = copy.deepcopy(raw)
wrong["groups"][0]["files"][0]["product"] = "ortho"
wrong["groups"][0]["files"][1]["colour"] = "red"
for problem in summaries.check(wrong):
    print("problem:", problem[:130])
check(wrong)                                        # load turns the same problems into an error, and blocks

**6. The registry.** Read the sample's registry and the real one ([`data_acquisition/sites.yaml`](../../../../data_acquisition/sites.yaml)) with `summaries.load_sites`. How many sites does each have? Extract the `name` of every site of the real one as a list and as a dict by slug, and list its regions. Then break a copy of the sample's registry (a CRS without `EPSG:`, a `land_status` that is not null or an entry, a site `contains` one that does not exist) and run `check_sites`. Finally, which sites does the reference summary use (`used_sites`)?

> **From the chapter.** §5: the registry functions. [Chapter 5](../../05_data_governance.md) explains the manual fields.
>
> **Code.** `summaries.load_sites`, `summaries.check_sites` and `summaries.used_sites`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
sample = summaries.load_sites()                     # BEAVER_SITES: the sample's registry
real = summaries.load_sites(ROOT / "data_acquisition/sites.yaml")
print(len(sample), "site in the sample's registry;", len(real), "in the real one")
names = [site["name"] for site in real.values()]
by_slug = {slug: site["name"] for slug, site in real.items()}
regions = [slug for slug, site in real.items() if site.get("level") == "region"]
print(names[:2], "...")
print(by_slug["rumney_ranch"])
print(len(regions), "regions:", regions)

broken = copy.deepcopy(sample)
broken["training_pond"]["crs"] = "32612"
broken["training_pond"]["land_status"] = "private"
broken["other"] = {"name": "Other", "crs": "EPSG:32612", "contains": ["nowhere"]}
for problem in summaries.check_sites(broken):
    print("problem:", problem)
print("used by the reference summary:", summaries.used_sites(summaries.load(str(
    ROOT / "docs/manual/solutions/training_pond.yaml"))))

**7. Anchors.** YAML lets you write a value once (`&name`) and reuse it (`*name`). Load a short text that does, and see what `yaml.safe_load` returns. Then run `summaries.anchors` on it, and on a text with `5 * 3 & more` in a note. Why must `anchors` look at the text, not the loaded dictionary?

> **From the chapter.** §5: a summary must write every value where it applies, so YAML anchors are refused.
>
> **Code.** `summaries.anchors`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py). `yaml.safe_load` is the YAML library's.

In [ ]:
with_anchor = "base: &common\n  platform: UAV\nfile_a: *common\n"
print(yaml.safe_load(with_anchor))
print(summaries.anchors(with_anchor))
print(summaries.anchors("note: 5 * 3 & more\nurl: http://x/*y\n"))
# The loaded dictionary has file_a's platform written out, as if the alias had never been there: once parsed,
# nothing shows it was used. So the check reads the raw text. A note with * or & is not an alias: they must stand
# alone after a space, colon, dash or bracket.

**8. A file's time.** For every data file of the reference summary: `resolve` it with its group, compute `summaries.epoch`, and print the path, start, end and label. Then compute `epoch` for a day, a month, a year, a range and an unknown date, and make it fail five ways.

> **From the chapter.** §5, on `epoch`: a file's time is a date with its precision, or a start and an end.
>
> **Code.** `summaries.resolve` and `summaries.epoch`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
group = s["groups"][0]
for entry in group["files"]:
    if entry.get("product"):                          # reports and metadata have no time of their own
        start, end, label = summaries.epoch(summaries.resolve(group, entry))
        print(f"{entry['path']:28s} {start}  {end}  {label}")

d = datetime.date
for f in ({"date": d(2023, 7, 12)}, {"date": d(2022, 8, 1), "date_precision": "month"},
          {"date": 2019, "date_precision": "year"}, {"start_date": d(2017, 9, 7), "end_date": d(2018, 9, 25)},
          {"date_precision": "unknown", "start_date": d(2010, 1, 1), "end_date": d(2015, 12, 31)}):
    print(f, "->", summaries.epoch(f))
for f in ({}, {"date": "12 July"}, {"date": d(2023, 7, 1), "date_precision": "week"},
          {"start_date": d(2020, 1, 1), "end_date": d(2019, 1, 1)}, {"date_precision": "unknown"}):
    try:
        summaries.epoch(f)
    except ValueError as e:
        print(f, "-> ValueError:", str(e)[:75])

**9. A layer's time.** A vector file can hold one layer per date, named by it. With `summaries.layer_epoch` and the pattern `_(\d{4})$` (year), find the date of the layers `Dams_2010`, `Dams_2013_old`, `notes` and `Dams_0000`. Then date `D26Jul_2018-07-26` by day and `x_2023-07` by month, and try `x_2023-02-30`. When does it answer None?

> **From the chapter.** §5, and [chapter 6](../../06_what_the_pipeline_needs.md), §3: a vector file can hold one date per layer, named in the layer.
>
> **Code.** `summaries.layer_epoch`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
spec = {"pattern": r"_(\d{4})$", "date_precision": "year"}
for name in ("Dams_2010", "Dams_2013_old", "notes", "Dams_0000"):
    print(f"{name:14s}", summaries.layer_epoch(name, spec))
day = {"pattern": r"_(\d{4}-\d{2}-\d{2})$", "date_precision": "day"}
month = {"pattern": r"_(\d{4}-\d{2})$", "date_precision": "month"}
print(summaries.layer_epoch("D26Jul_2018-07-26", day), summaries.layer_epoch("x_2023-07", month))
print(summaries.layer_epoch("x_2023-02-30", day))
# None when the pattern does not match (`notes`, and `Dams_2013_old` because the pattern ends at `$`), or when what it
# captures is not a real date (year 0, 30 February). The builder reports those layers as unmatched_layers.

**10. Who wrote it.** Call `summaries.author` on the reference summary. Then on copies with `reviewed` set, and with a person as author. What does `summaries.by_claude` say about `claude`, `claude (claude-opus-5-5)`, `Claudia Rossi` and `Federico`?

> **From the chapter.** §4: a person sets `reviewed`, and the catalogue's labels follow it.
>
> **Code.** `summaries.author` and `summaries.by_claude`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py).

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
print("as written:", summaries.author(s))
reviewed = copy.deepcopy(s)
reviewed["reviewed"] = {"by": "Federico Oliva", "date": datetime.date(2026, 11, 3)}
print("reviewed:  ", summaries.author(reviewed))
by_person = copy.deepcopy(s)
by_person["authored"] = {"by": "Federico Oliva", "date": datetime.date(2026, 10, 1)}
print("by a person:", summaries.author(by_person))
print([summaries.by_claude(who) for who in ("claude", "claude (claude-opus-5-5)", "Claudia Rossi", "Federico")])
# `reviewed` wins: once a person has checked the summary, its values count as manual. by_claude only looks at
# whether the name starts with lower-case "claude".

**11. Evidence in code.** In a scratch folder with a copy of the orthomosaic and a sidecar, what does `evidence.entries` list? On the sample's files, plus a broken `.tif` and an unknown `.bin`, what does `evidence.describe_file` return? Then run `evidence.gather` on the sample, and on a folder of 10 photos: why is that one entry, when the sample's 6 photos are six?

> **From the chapter.** §2: what evidence is gathered, and §5: its functions.
>
> **Code.** `evidence.entries`, `evidence.describe_file`, `evidence.gather` and `evidence.PHOTO_FOLDER`, in [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py).

In [ ]:
scratch = Path(tempfile.mkdtemp())
shutil.copyfile(DATASET / "ortho_2023-07-12.tif", scratch / "ortho.tif")
(scratch / "ortho.tif.aux.xml").write_text("<PAMDataset/>")
(scratch / "bad.tif").write_bytes(b"not a tiff")
(scratch / "data.bin").write_bytes(b"1")
print("entries:", [p.name for p in evidence.entries(scratch)])           # the sidecar is left out

for name in ("ortho_2023-07-12.tif", "dams_2023-07-12.gpkg", "metadata.xml", "README.txt",
             "photos_2023-07-12/100MEDIA/DJI_0001.JPG"):
    print(f"{name:40s} keys: {list(evidence.describe_file(DATASET / name))}")
print("a broken file:", str(evidence.describe_file(scratch / "bad.tif"))[:60], "...")     # an error is evidence too
print("an unknown type:", evidence.describe_file(scratch / "data.bin"))
# A single photo gets nothing: describe_file has no branch for it (finding 10 in course_findings.md).

found = evidence.gather(SAMPLE, ["repo/*"])
print(list(found), "|", [f["folder"] for f in found["folders"]], "|", len(found["folders"][0]["files"]), "entries")
print("a pattern matching nothing:", evidence.gather(SAMPLE, ["nothing/*"])["folders"])

photos_root = Path(tempfile.mkdtemp())
folder = photos_root / "survey" / "100MEDIA"
folder.mkdir(parents=True)
originals = sorted((DATASET / "photos_2023-07-12/100MEDIA").glob("*.JPG"))
for i in range(evidence.PHOTO_FOLDER):
    shutil.copyfile(originals[i % len(originals)], folder / f"P{i:02d}.JPG")
print(evidence.gather(photos_root, ["survey"])["folders"][0]["files"])
# From PHOTO_FOLDER (10) photos up, a folder is one entry: count, size, first and last photo's EXIF. Thousands of
# photos would otherwise fill the evidence; below that, each photo is listed, with nothing read from it.

**12. The prompt and the check in code.** Print `describe.registry_lines()`. Write a prompt into a scratch folder with `describe.write_prompt`: which files appear, and what did it fill in? Then run `describe.check_summary` on the reference summary, on a copy that names a missing file and leaves the DSM out, and on one with an unknown product. What does it return each time?

> **From the chapter.** §3: the prompt and the check, and §5: their functions.
>
> **Code.** `describe.registry_lines`, `describe.write_prompt` and `describe.check_summary`, in [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py); `evidence.gather` in [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py).

In [ ]:
found = evidence.gather(SAMPLE, ["repo/*"])
print(describe.registry_lines())
out = Path(tempfile.mkdtemp())
prompt = describe.write_prompt(found, "training_pond", "dataone", [], out)
print(sorted(p.name for p in out.iterdir()), "|", prompt.read_text().splitlines()[0])
print("lists the site:", "`training_pond`" in prompt.read_text(), "| holds the rules:", "never" in prompt.read_text())

reference = ROOT / "docs/manual/solutions/training_pond.yaml"
print("returns", describe.check_summary(SAMPLE, str(reference)))
raw = yaml.safe_load(reference.read_text())
raw["groups"][0]["files"] = [f for f in raw["groups"][0]["files"] if f["path"] != "dsm_2023-07-12.tif"]
raw["groups"][0]["files"].append({"path": "ghost.tif", "product": "chm", "date": datetime.date(2023, 7, 12)})
path = Path(tempfile.mkdtemp()) / "summary.yaml"
path.write_text(yaml.safe_dump(raw, sort_keys=False))
print("returns", describe.check_summary(SAMPLE, str(path)))         # the build's: ghost.tif, the DSM, README -> dsm
raw["groups"][0]["files"][0]["product"] = "ortho"
path.write_text(yaml.safe_dump(raw, sort_keys=False))
print("returns", describe.check_summary(SAMPLE, str(path))) # 1: load stops first, so the build never ran
# It returns the number of problems, and the command exits with 1 when there are any. Careful when writing YAML
# from Python: safe_dump writes an alias for an object used twice (one date on two files); anchors() refuses it.

**13. (Optional) Draft with Claude.** Run the prompt in a Claude Code session on the sample. The prompt writes into [`data_acquisition/summaries/`](../../../../data_acquisition/summaries) and may add a site to [`data_acquisition/sites.yaml`](../../../../data_acquisition/sites.yaml): delete the draft and undo any registry change afterwards (`git status`, `git restore`).

> **From the chapter.** §3: drafting in a new Claude Code session.
>
> **Code.** `prompt.md` from exercise 1, in a Claude Code session. It may write into [`data_acquisition/summaries/`](../../../../data_acquisition/summaries) and [`data_acquisition/sites.yaml`](../../../../data_acquisition/sites.yaml).

In [ ]:
# Done in a Claude Code session, not here. Afterwards, check what it changed:
status = subprocess.run(["git", "status", "--short"], cwd=ROOT, capture_output=True, text=True).stdout
print(status or "nothing changed")

**14. Find it in the code.** Where are the drafting rules, and which function decides what evidence a raster gets?

> **From the chapter.** §3 for the rules, §2 for the evidence.
>
> **Code.** Look in [`data_acquisition/catalog/describe.py`](../../../../data_acquisition/catalog/describe.py) and [`data_acquisition/catalog/evidence.py`](../../../../data_acquisition/catalog/evidence.py) (imported above as `describe` and `evidence`).

In [ ]:
import inspect
print("rules: describe.RULES in", inspect.getsourcefile(describe), "line",
      next(n for n, l in enumerate(inspect.getsource(describe).splitlines(), 1) if l.startswith("RULES")))
print(inspect.getsource(evidence._raster))